# Phase 1 - Logit Lens

Goal: track where person-bearing verb information appears. This notebook now has two analyses: target-token rank and person-form margin.

## 1. Setup

In [1]:
from pathlib import Path
import importlib
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.dataset import get_pilot
import core.logit_lens as logit_lens
logit_lens = importlib.reload(logit_lens)

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 100)

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

d:\Projects\ProDropLens\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: d:\Projects\ProDropLens


## 2. Load Model

The Turkish GPT-2 checkpoint is loaded through the GPT-2 TransformerLens architecture bridge. Tokenization stays with the HF Turkish tokenizer.

In [2]:
MODEL_NAME = "ytu-ce-cosmos/turkish-gpt2"
model, tokenizer = logit_lens.load_lens_model(MODEL_NAME)
print(f"Layers: {model.cfg.n_layers}")
print(f"d_model: {model.cfg.d_model}")
print(f"d_vocab: {model.cfg.d_vocab}")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 8706.27it/s]


Loaded pretrained model gpt2 into HookedTransformer
Layers: 12
d_model: 768
d_vocab: 50257


## 3. Original Target-Token Specs

In [3]:
pilot_pairs = get_pilot()
specs_df = logit_lens.build_logit_lens_specs(tokenizer, pilot_pairs)
display(specs_df[[
    "pair_id", "variant", "person", "text", "target_token", "target_token_id",
    "read_pos", "compare_mode", "sentence_tokens", "sentence_token_ids",
]])

,pair_id,variant,person,text,target_token,target_token_id,read_pos,compare_mode,sentence_tokens,sentence_token_ids
0,pilot_gitmek_simdiki_1s,overt,1s,Ben her gün okula gidiyorum.,gidiyorum,23301,3,single_token_logit_diff,Ben | her | gün | okula | gidiyorum | .,2524 | 556 | 543 | 9735 | 23301 | 14
1,pilot_gitmek_simdiki_1s,prodrop,1s,Her gün okula gidiyorum.,gidiyorum,23301,2,single_token_logit_diff,Her | gün | okula | gidiyorum | .,3024 | 543 | 9735 | 23301 | 14
2,pilot_gitmek_simdiki_2s,overt,2s,Sen her gün okula gidiyorsun.,sun,833,4,suffix_probe_plus_sequence_logprob,Sen | her | gün | okula | gidiyor | sun | .,5425 | 556 | 543 | 9735 | 6215 | 833 | 14
3,pilot_gitmek_simdiki_2s,prodrop,2s,Her gün okula gidiyorsun.,sun,833,3,suffix_probe_plus_sequence_logprob,Her | gün | okula | gidiyor | sun | .,3024 | 543 | 9735 | 6215 | 833 | 14
4,pilot_gitmek_simdiki_3s,overt,3s,O her gün okula gidiyor.,gidiyor,6215,3,self_contrast_baseline,O | her | gün | okula | gidiyor | .,47 | 556 | 543 | 9735 | 6215 | 14
5,pilot_gitmek_simdiki_3s,prodrop,3s,Her gün okula gidiyor.,gidiyor,6215,2,self_contrast_baseline,Her | gün | okula | gidiyor | .,3024 | 543 | 9735 | 6215 | 14
6,pilot_gitmek_simdiki_1p,overt,1p,Biz her gün okula gidiyoruz.,gidiyoruz,19015,3,single_token_logit_diff,Biz | her | gün | okula | gidiyoruz | .,3722 | 556 | 543 | 9735 | 19015 | 14
7,pilot_gitmek_simdiki_1p,prodrop,1p,Her gün okula gidiyoruz.,gidiyoruz,19015,2,single_token_logit_diff,Her | gün | okula | gidiyoruz | .,3024 | 543 | 9735 | 19015 | 14
8,pilot_gitmek_simdiki_2p,overt,2p,Siz her gün okula gidiyorsunuz.,sunuz,1711,4,suffix_probe_plus_sequence_logprob,Siz | her | gün | okula | gidiyor | sunuz | .,6050 | 556 | 543 | 9735 | 6215 | 1711 | 14
9,pilot_gitmek_simdiki_2p,prodrop,2p,Her gün okula gidiyorsunuz.,sunuz,1711,3,suffix_probe_plus_sequence_logprob,Her | gün | okula | gidiyor | sunuz | .,3024 | 543 | 9735 | 6215 | 1711 | 14


## 4. Run Original Logit Lens

In [4]:
lens_df = logit_lens.run_logit_lens(model, tokenizer, specs_df, top_k=5)
summary_df = logit_lens.summarize_rank_transitions(lens_df)
comparison_df = logit_lens.compare_overt_prodrop(lens_df)

specs_df.to_csv(RESULTS_DIR / "phase1_logit_lens_specs.csv", index=False)
lens_df.to_csv(RESULTS_DIR / "phase1_logit_lens_layers.csv", index=False)
summary_df.to_csv(RESULTS_DIR / "phase1_transition_summary.csv", index=False)
comparison_df.to_csv(RESULTS_DIR / "phase1_overt_prodrop_comparison.csv", index=False)

print(f"lens rows: {len(lens_df)}")

lens rows: 144


## 5. Target Rank Trajectories

In [5]:
rank_pivot = lens_df.pivot_table(
    index="layer",
    columns=["person", "variant"],
    values="target_rank",
    aggfunc="first",
)
display(rank_pivot)

person     1p            1s            2p            2s            3p            3s        
variant overt prodrop overt prodrop overt prodrop overt prodrop overt prodrop overt prodrop
layer                                                                                      
0         573     937   174     343     3       3     4       4   172     345    56      61
1        1009    1868   306     865     1       1     3       2   597     799    48      64
2         807    1656   317     880     1       1     3       2  1313    1810    99     137
3         573    1076   113     340     1       1     4       3   954    1283    55      74
4         317     743   124     368     1       1     2       6   538     882    44      59
5         197     441    84     300     5       5     6       8   350     619    33      40
6         189     469    42     189     1       3     6      12   281     499    30      44
7         147     295    33     116     1       1     3       8   230     456    22      44
8          75     232    11      55     1       1     2       4    75     235     9      18
9          14      51     3      10     1       1     2       4    16      79     3       6
10         10      29     2       8     1       1     1       3    10      41     1       7
11          2      10     2       5     2       2     1       6     2      21     1       6

## 6. Target Rank Transition Summary

In [6]:
display(summary_df[[
    "pair_id", "variant", "person", "target_form", "target_token", "compare_mode",
    "initial_rank", "final_rank", "best_rank", "best_rank_layer",
    "critical_from_layer", "critical_to_layer",
    "first_layer_rank_le_10", "first_layer_rank_le_5", "first_layer_rank_le_1",
]])

,pair_id,variant,person,target_form,target_token,compare_mode,initial_rank,final_rank,best_rank,best_rank_layer,critical_from_layer,critical_to_layer,first_layer_rank_le_10,first_layer_rank_le_5,first_layer_rank_le_1
0,pilot_gitmek_simdiki_1p,overt,1p,gidiyoruz,gidiyoruz,single_token_logit_diff,573,2,2,11,8,9,10.0,11.0,NaN
1,pilot_gitmek_simdiki_1p,prodrop,1p,gidiyoruz,gidiyoruz,single_token_logit_diff,937,10,10,11,8,9,11.0,NaN,NaN
2,pilot_gitmek_simdiki_1s,overt,1s,gidiyorum,gidiyorum,single_token_logit_diff,174,2,2,10,8,9,9.0,9.0,NaN
3,pilot_gitmek_simdiki_1s,prodrop,1s,gidiyorum,gidiyorum,single_token_logit_diff,343,5,5,11,8,9,9.0,11.0,NaN
4,pilot_gitmek_simdiki_2p,overt,2p,gidiyorsunuz,sunuz,suffix_probe_plus_sequence_logprob,3,2,1,1,5,6,0.0,0.0,1.0
5,pilot_gitmek_simdiki_2p,prodrop,2p,gidiyorsunuz,sunuz,suffix_probe_plus_sequence_logprob,3,2,1,1,0,1,0.0,0.0,1.0
6,pilot_gitmek_simdiki_2s,overt,2s,gidiyorsun,sun,suffix_probe_plus_sequence_logprob,4,1,1,10,6,7,0.0,0.0,10.0
7,pilot_gitmek_simdiki_2s,prodrop,2s,gidiyorsun,sun,suffix_probe_plus_sequence_logprob,4,6,2,1,7,8,0.0,0.0,NaN
8,pilot_gitmek_simdiki_3p,overt,3p,gidiyorlar,gidiyorlar,single_token_logit_diff,172,2,2,11,8,9,10.0,11.0,NaN
9,pilot_gitmek_simdiki_3p,prodrop,3p,gidiyorlar,gidiyorlar,single_token_logit_diff,345,21,21,11,8,9,NaN,NaN,NaN


## 7. Original Overt vs. Pro-Drop Final Layer

In [7]:
layer11_comparison = comparison_df[comparison_df["layer"] == 11]
display(layer11_comparison[[
    "person", "target_form", "target_token", "compare_mode", "layer",
    "overt_target_rank", "prodrop_target_rank", "target_rank_diff_prodrop_minus_overt",
    "overt_target_log_prob", "prodrop_target_log_prob",
]])

,person,target_form,target_token,compare_mode,layer,overt_target_rank,prodrop_target_rank,target_rank_diff_prodrop_minus_overt,overt_target_log_prob,prodrop_target_log_prob
11,1p,gidiyoruz,gidiyoruz,single_token_logit_diff,11,2,10,8,-2.446491,-3.902147
23,1s,gidiyorum,gidiyorum,single_token_logit_diff,11,2,5,3,-2.093292,-2.934170
35,2p,gidiyorsunuz,sunuz,suffix_probe_plus_sequence_logprob,11,2,2,0,-1.427097,-1.996912
47,2s,gidiyorsun,sun,suffix_probe_plus_sequence_logprob,11,1,6,5,-1.670042,-3.089511
59,3p,gidiyorlar,gidiyorlar,single_token_logit_diff,11,2,21,19,-2.544066,-4.936614
71,3s,gidiyor,gidiyor,self_contrast_baseline,11,1,6,5,-2.018838,-3.014892


## 8. Person-Form Margin Specs

For each context, score all six person forms from the same verb and tense. This asks whether the correct person form separates from competing forms, not just whether one token becomes high-rank.

In [8]:
person_specs_df = logit_lens.build_person_form_specs(tokenizer, pilot_pairs)
display(person_specs_df[[
    "pair_id", "variant", "actual_person", "candidate_person", "is_correct",
    "prefix_text", "candidate_form", "candidate_tokens", "candidate_read_positions",
]])

,pair_id,variant,actual_person,candidate_person,is_correct,prefix_text,candidate_form,candidate_tokens,candidate_read_positions
0,pilot_gitmek_simdiki_1s,overt,1s,1s,True,Ben her gün okula,gidiyorum,gidiyorum,3
1,pilot_gitmek_simdiki_1s,overt,1s,2s,False,Ben her gün okula,gidiyorsun,gidiyor | sun,3 | 4
2,pilot_gitmek_simdiki_1s,overt,1s,3s,False,Ben her gün okula,gidiyor,gidiyor,3
3,pilot_gitmek_simdiki_1s,overt,1s,1p,False,Ben her gün okula,gidiyoruz,gidiyoruz,3
4,pilot_gitmek_simdiki_1s,overt,1s,2p,False,Ben her gün okula,gidiyorsunuz,gidiyor | sunuz,3 | 4
...,...,...,...,...,...,...,...,...,...
67,pilot_gitmek_simdiki_3p,prodrop,3p,2s,False,Her gün okula,gidiyorsun,gidiyor | sun,2 | 3
68,pilot_gitmek_simdiki_3p,prodrop,3p,3s,False,Her gün okula,gidiyor,gidiyor,2
69,pilot_gitmek_simdiki_3p,prodrop,3p,1p,False,Her gün okula,gidiyoruz,gidiyoruz,2
70,pilot_gitmek_simdiki_3p,prodrop,3p,2p,False,Her gün okula,gidiyorsunuz,gidiyor | sunuz,2 | 3


## 9. Run Person-Form Margin Lens

Multi-token forms are scored with teacher-forced layer-wise sequence log-probability. `sequence_log_prob_mean` is used for the main margin to reduce token-count bias.

In [9]:
person_scores_df = logit_lens.run_person_form_lens(model, tokenizer, person_specs_df)
person_margins_df = logit_lens.summarize_person_margins(person_scores_df)
person_margin_summary_df = logit_lens.summarize_person_margin_transitions(person_margins_df)

person_specs_df.to_csv(RESULTS_DIR / "phase1_person_form_specs.csv", index=False)
person_scores_df.to_csv(RESULTS_DIR / "phase1_person_form_scores.csv", index=False)
person_margins_df.to_csv(RESULTS_DIR / "phase1_person_margin_layers.csv", index=False)
person_margin_summary_df.to_csv(RESULTS_DIR / "phase1_person_margin_transition_summary.csv", index=False)

print(f"person score rows: {len(person_scores_df)}")

person score rows: 864


## 10. Candidate Scores Example

Example context: `Ben her g?n okula ...`. Positive separation should mean `gidiyorum` beats all other person forms.

In [10]:
example_scores = person_scores_df[
    (person_scores_df["pair_id"] == "pilot_gitmek_simdiki_1s")
    & (person_scores_df["variant"] == "overt")
    & (person_scores_df["layer"].isin([0, 9, 11]))
]
display(example_scores[[
    "layer", "candidate_person", "candidate_form", "sequence_log_prob_mean",
    "candidate_rank_mean", "token_vocab_ranks",
]])

,layer,candidate_person,candidate_form,sequence_log_prob_mean,candidate_rank_mean,token_vocab_ranks
0,0,1s,gidiyorum,-562.167847,4,174
9,9,1s,gidiyorum,-2.278668,1,3
11,11,1s,gidiyorum,-2.093292,1,2
12,0,2s,gidiyorsun,-348.839882,2,42 | 5
21,9,2s,gidiyorsun,-3.161831,3,6 | 5
23,11,2s,gidiyorsun,-2.957348,3,6 | 4
24,0,3s,gidiyor,-560.304565,3,42
33,9,3s,gidiyor,-3.264731,4,6
35,11,3s,gidiyor,-2.789996,2,6
36,0,1p,gidiyoruz,-563.702881,6,542


## 11. Person Margin by Layer

`person_margin_mean = score(correct form) - score(best wrong form)`. Positive values mean the correct person form is selected among the six candidates.

In [11]:
margin_pivot = person_margins_df.pivot_table(
    index="layer",
    columns=["actual_person", "variant"],
    values="person_margin_mean",
    aggfunc="first",
)
display(margin_pivot)

actual_person          1p                      1s                    2p                  2s                    3p                      3s            
variant             overt     prodrop       overt     prodrop     overt   prodrop     overt   prodrop       overt     prodrop       overt     prodrop
layer                                                                                                                                                
0             -219.648308 -279.707920 -213.770889 -278.345676  0.762001  0.452805 -0.461781 -0.452805 -235.499542 -278.347019 -243.591999 -276.058933
1             -192.381196 -246.516012 -156.057882 -245.229543  1.006517  0.616412 -0.411256 -0.616412 -185.207059 -245.107839 -218.794181 -241.809255
2              -46.562199  -90.871567  -15.867153  -89.759370  0.655273  0.319859 -0.261574 -0.319859  -65.523391  -91.058915  -73.552930  -86.834214
3              -72.394809 -107.343548  -58.667257 -105.452648  0.659809  0.215072 -0.313790 -0.215072  -96.471299 -107.638744 -111.549795 -103.391292
4              -58.723247  -88.650334  -35.079750  -87.591252  0.897623  0.437893 -0.300098 -0.437893  -67.063256  -88.968114 -117.743574  -85.146367
5               -3.760193   -4.705408   -2.353221   -4.069733  0.639859  0.339482  0.007609 -0.339482   -4.430830   -5.220517  -31.067645   -1.191795
6               -3.879027   -5.089154   -1.955402   -3.509761  1.120678  0.871731 -0.523218 -0.871731   -3.734069   -5.207933   -0.901451   -1.765844
7               -4.112915   -5.106027   -2.029396   -3.715478  1.541039  1.189844 -1.038529 -1.189844   -3.942961   -5.874421   -0.975308   -2.409547
8               -3.750636   -5.310889   -1.222327   -3.059512  1.334590  0.758165 -0.706688 -0.758165   -2.586910   -5.329213    0.673537   -1.856715
9               -1.933008   -4.145324    0.580728   -1.882497  1.383761  0.845429 -0.598381 -0.845429   -2.338900   -4.711768    1.326668   -1.161185
10              -0.666435   -2.445633    1.004230   -0.948848  0.825194  0.296781  0.310224 -0.296781   -1.470226   -3.011592    1.649709   -0.731618
11               0.056070   -1.396245    0.696704   -0.428268  0.590810  0.428268  0.333869 -0.546298   -0.681884   -2.430711    2.208215   -0.508988

## 12. Person Margin Transition Summary

In [12]:
display(person_margin_summary_df[[
    "actual_person", "variant", "correct_candidate_form",
    "initial_margin", "final_margin", "best_margin", "best_margin_layer",
    "first_positive_margin_layer", "critical_from_layer", "critical_to_layer",
    "final_correct_form_rank_mean", "final_selected_person_mean",
    "final_selected_form_mean", "final_selected_is_correct_mean",
]])

,actual_person,variant,correct_candidate_form,initial_margin,final_margin,best_margin,best_margin_layer,first_positive_margin_layer,critical_from_layer,critical_to_layer,final_correct_form_rank_mean,final_selected_person_mean,final_selected_form_mean,final_selected_is_correct_mean
0,1p,overt,gidiyoruz,-219.648308,0.056070,0.056070,11,11.0,1,2,1,1p,gidiyoruz,True
1,1p,prodrop,gidiyoruz,-279.707920,-1.396245,-1.396245,11,NaN,1,2,5,2p,gidiyorsunuz,False
2,1s,overt,gidiyorum,-213.770889,0.696704,1.004230,10,9.0,1,2,1,1s,gidiyorum,True
3,1s,prodrop,gidiyorum,-278.345676,-0.428268,-0.428268,11,NaN,1,2,2,2p,gidiyorsunuz,False
4,2p,overt,gidiyorsunuz,0.762001,0.590810,1.541039,7,0.0,5,6,1,2p,gidiyorsunuz,True
5,2p,prodrop,gidiyorsunuz,0.452805,0.428268,1.189844,7,0.0,5,6,1,2p,gidiyorsunuz,True
6,2s,overt,gidiyorsun,-0.461781,0.333869,0.333869,11,5.0,9,10,1,2s,gidiyorsun,True
7,2s,prodrop,gidiyorsun,-0.452805,-0.546298,-0.215072,3,NaN,9,10,4,2p,gidiyorsunuz,False
8,3p,overt,gidiyorlar,-235.499542,-0.681884,-0.681884,11,NaN,1,2,2,3s,gidiyor,False
9,3p,prodrop,gidiyorlar,-278.347019,-2.430711,-2.430711,11,NaN,1,2,6,2p,gidiyorsunuz,False


## 13. Final-Layer Person Selection

In [13]:
final_person_margins = person_margins_df[person_margins_df["layer"] == 11]
display(final_person_margins[[
    "actual_person", "variant", "correct_candidate_form", "person_margin_mean",
    "correct_form_rank_mean", "selected_person_mean", "selected_form_mean",
    "selected_is_correct_mean", "best_wrong_person_mean", "best_wrong_form_mean",
]])

,actual_person,variant,correct_candidate_form,person_margin_mean,correct_form_rank_mean,selected_person_mean,selected_form_mean,selected_is_correct_mean,best_wrong_person_mean,best_wrong_form_mean
11,1p,overt,gidiyoruz,0.056070,1,1p,gidiyoruz,True,1s,gidiyorum
23,1p,prodrop,gidiyoruz,-1.396245,5,2p,gidiyorsunuz,False,2p,gidiyorsunuz
35,1s,overt,gidiyorum,0.696704,1,1s,gidiyorum,True,3s,gidiyor
47,1s,prodrop,gidiyorum,-0.428268,2,2p,gidiyorsunuz,False,2p,gidiyorsunuz
59,2p,overt,gidiyorsunuz,0.590810,1,2p,gidiyorsunuz,True,3s,gidiyor
71,2p,prodrop,gidiyorsunuz,0.428268,1,2p,gidiyorsunuz,True,1s,gidiyorum
83,2s,overt,gidiyorsun,0.333869,1,2s,gidiyorsun,True,3s,gidiyor
95,2s,prodrop,gidiyorsun,-0.546298,4,2p,gidiyorsunuz,False,2p,gidiyorsunuz
107,3p,overt,gidiyorlar,-0.681884,2,3s,gidiyor,False,3s,gidiyor
119,3p,prodrop,gidiyorlar,-2.430711,6,2p,gidiyorsunuz,False,2p,gidiyorsunuz


## 14. Updated Interpretation

- Target-token rank alone still shows whole-token targets improving around late layers, but it mixes subject evidence with local lexical continuation.
- Person-form margin is the stricter subject-selection metric: it compares the correct form against all five competing person forms in the same context.
- In overt contexts, `gidiyorum`, `gidiyorsun`, `gidiyorsunuz`, and `gidiyor` become the selected form by the final layer. `gidiyoruz` only barely wins, while `gidiyorlar` does not beat singular `gidiyor` in this pilot.
- In pro-drop contexts, most target persons do not become selected because the prefix is genuinely person-ambiguous. The frequent/default continuation often remains `gidiyorsunuz` or `gidiyor`.
- Early wins for suffix-split forms are consistent with lexical continuation from `gidiyor` rather than a clean subject mechanism. Treat these rows as controls for tokenization/frequency effects.